# Fisiere: text, CSV, imagini (tema)

Apasa **Copy to Drive** (sus, in stanga) inainte sa scrii ceva aici, altfel tot ce faci dispare cand inchizi tab-ul.

Ruleaza prima celula de mai jos, o singura data, ca sa aduci utilitarele cursului.

In [ ]:
import os, urllib.request
if not os.path.exists('onia.py'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/alexbirs27/iLoveBeer/main/cursuri/colab/onia.py', 'onia.py')
from onia import *
descarca('date/elevi.csv', 'date/09_forme.png', 'date/09_mic.png', 'anim_imagini.py')
from anim_imagini import *

Exercitiile de acasa. Testele vizibile iti arata daca ai inteles enuntul, cele **ascunse** verifica si cazurile limita. La final, `scor()` aduna tot.

Testele creeaza singure fisiere temporare (`pe_fisier`, definita in celula de mai jos), apeleaza functia ta pe ele si le sterg. Tu scrii doar functia care primeste **calea**. Ruleaz-o o data.

In [ ]:
import os, tempfile

def pe_fisier(f, continut, *alte, nume='a.txt'):
    # scrie 'continut' intr-un fisier temporar, apeleaza f(cale, *alte), sterge fisierul
    with tempfile.TemporaryDirectory() as d:
        cale = os.path.join(d, nume)
        with open(cale, 'w', encoding='utf-8', newline='') as g:
            g.write(continut)
        return f(cale, *alte)

import numpy as np

### 1. Filtreaza linii

`linii_cu(cale, cuvant)`: intoarce lista liniilor din fisier care **contin** textul `cuvant` (cautare exacta, litere mari si mici conteaza), in ordinea din fisier si **fara** `\n` la capat.

In [ ]:
def linii_cu(cale, cuvant):
    pass  # TODO

In [ ]:
verifica('linii_cu', lambda text, cuv: pe_fisier(linii_cu, text, cuv),
         [(('mere rosii\npere\nmere verzi\n', 'mere'), ['mere rosii', 'mere verzi']), (('a\nb\n', 'c'), [])],
         ascunse=[(('Mere\nmere\n', 'mere'), ['mere']), (('', 'x'), []), (('ab\nba\n\nb', 'b'), ['ab', 'ba', 'b']),
                  (('  x  \ny\n', 'x'), ['  x  '])],
         puncte=1, indiciu='append pe linie.strip(\'\\n\') (nu strip() simplu: nu vrei sa pierzi spatiile de la capete)')

### 2. Diacritice

Fisierul e scris in `utf-8` si contine text romanesc. `numara_diacritice(cale)` intoarce de cate ori apar in el literele cu semne diacritice romanesti, mici sau mari: `a` cu semn (`\u0103`, `\u00e2`), `i` cu semn (`\u00ee`), `s` cu virgula (`\u0219`), `t` cu virgula (`\u021b`) si majusculele lor (`\u0102`, `\u00c2`, `\u00ce`, `\u0218`, `\u021a`). Ca sa ramana codul in ASCII, scrie literele in program tot cu escape-uri, ca in enunt.

In [ ]:
def numara_diacritice(cale):
    pass  # TODO: deschide cu encoding='utf-8'

In [ ]:
verifica('numara_diacritice', lambda text: pe_fisier(numara_diacritice, text),
         [(('Ana are mere \u0219i pere\n',), 1), (('\u021b\u0103ranul \u00eenv\u0103\u021b\u0103tor\n',), 6)],
         ascunse=[(('',), 0), (('fara diacritice\n',), 0), (('\u0102\u00c2\u00ce\u0218\u021a\n\u0103\u00e2\u00ee\u0219\u021b',), 10)],
         puncte=1, indiciu='open(cale, encoding=\'utf-8\'); ch in \'...\' pentru fiecare caracter')

### 3. Valori lipsa pe coloane

`lipsuri_pe_coloana(cale)`: fisierul e CSV cu antet. Intoarce un dictionar `{nume_coloana: cate celule sunt goale}` (o celula goala e sirul vid, sau lipseste pentru ca randul e prea scurt). Dupa testele mici, verifica rezultatul si pe `date/elevi.csv`, comparat cu `pandas`.

In [ ]:
import csv


def lipsuri_pe_coloana(cale):
    pass  # TODO

In [ ]:
import pandas as pd

verifica('lipsuri_pe_coloana', lambda text: pe_fisier(lipsuri_pe_coloana, text, nume='a.csv'),
         [(('a,b,c\n1,,3\n,,\n4,5,\n',), {'a': 1, 'b': 2, 'c': 2}), (('x\n1\n2\n',), {'x': 0})],
         ascunse=[(('a,b\n',), {'a': 0, 'b': 0}), (('a,b\n1\n,2\n',), {'a': 1, 'b': 1}), (('a,b\n"x, y",\n',), {'a': 0, 'b': 1})],
         puncte=2, indiciu='DictReader da None pentru celulele care lipsesc complet: (r[col] or \'\')')

de_pandas = {k: int(v) for k, v in pd.read_csv('date/elevi.csv').isna().sum().items()}
verifica('lipsuri pe date/elevi.csv', lipsuri_pe_coloana, [(('date/elevi.csv',), de_pandas)], puncte=1)

### 4. Repara codul: numarul de note

`suma_note(cale)` trebuie sa adune notele dintr-un fisier cu linii de forma `nume,nota`. Pe fisiere care se termina cu `\n` (adica aproape toate) crapa. Animatia de mai jos arata problema pe un text; repara functia.

In [ ]:
%%vezi
text = 'Ana,9\nDan,7\n'
suma = 0
for linie in text.split('\n'):            #: split('\n') pastreaza un element gol dupa ultimul \n
    suma += int(linie.split(',')[1])

In [ ]:
def suma_note(cale):
    with open(cale, encoding='utf-8') as f:
        continut = f.read()
    suma = 0
    for linie in continut.split('\n'):
        suma += int(linie.split(',')[1])
    return suma

In [ ]:
verifica('suma_note', lambda text: pe_fisier(suma_note, text),
         [(('Ana,9\nDan,7\n',), 16), (('Eva,10',), 10)],
         ascunse=[(('',), 0), (('a,1\nb,2\nc,3\n',), 6), (('x,5\n',), 5)],
         puncte=1, indiciu='"a\\nb\\n".split("\\n") are 3 elemente, ultimul gol. Ce metoda nu lasa element gol?')

### 5. Repara codul: jurnalul

`adauga_in_jurnal(cale, mesaj)` trebuie sa adauge `mesaj` pe o linie noua la sfarsitul fisierului, fara sa piarda ce era deja acolo. Acum pierde tot si nici nu trece la linie noua. Sunt doua greseli.

In [ ]:
def adauga_in_jurnal(cale, mesaj):
    with open(cale, 'w', encoding='utf-8') as f:
        f.write(mesaj)

In [ ]:
def t_jurnal(mesaje):
    with tempfile.TemporaryDirectory() as d:
        cale = os.path.join(d, 'jurnal.txt')
        for m in mesaje:
            adauga_in_jurnal(cale, m)
        with open(cale, encoding='utf-8') as f:
            return f.read().splitlines()

verifica('adauga_in_jurnal', t_jurnal, [((['pornit', 'oprit'],), ['pornit', 'oprit']), ((['a'],), ['a'])],
         ascunse=[((['1', '2', '3'],), ['1', '2', '3']), ((['x', 'x'],), ['x', 'x'])],
         puncte=1, indiciu='ce mod de deschidere nu sterge fisierul? si write nu pune singur \\n')

### 6. Program: notele clasei

Program ca la concurs. Prima linie: `n`. Urmeaza `n` linii CSV de forma `nume,nota`. Numele poate contine virgula, caz in care apare intre ghilimele (`"Popescu, Ion",9`), iar nota poate lipsi (`Dan,`). Afiseaza pe prima linie media notelor care exista, cu **doua zecimale**, iar pe a doua numele elevului cu nota cea mai mare (la egalitate, primul din lista). Se garanteaza ca exista cel putin o nota.

Sfat: `csv.reader([linie])` citeste o singura linie deja in memorie, cu regulile CSV.

In [ ]:
import csv


def main():
    pass  # TODO

In [ ]:
verifica_program('note clasa', main,
                 [(['3', 'Ana,9', 'Dan,', 'Eva,7'], '8.00\nAna'),
                  (['2', '"Popescu, Ion",10', 'Ana,8'], '9.00\nPopescu, Ion')],
                 ascunse=[(['1', 'X,5'], '5.00\nX'), (['4', 'a,7', 'b,9', 'c,', 'd,9'], '8.33\nb'),
                          (['2', 'Ana,', 'Dan,6.5'], '6.50\nDan')],
                 puncte=2, indiciu='next(csv.reader([linie])) da lista [nume, nota]; max cu key= intoarce primul la egalitate')

### 7. Valideaza un submission

Inainte sa trimiti, verifici. `valideaza_submission(cale, n)` intoarce `True` doar daca fisierul are exact: antetul `subtaskID,datapointID,answer`; `n` randuri de date; pe fiecare rand exact 3 campuri, `datapointID` egal cu numarul de ordine al randului (1, 2, ..., n) si `answer` nevid. Altfel `False`.

In [ ]:
import csv


def valideaza_submission(cale, n):
    pass  # TODO

In [ ]:
H = 'subtaskID,datapointID,answer\n'
verifica('valideaza_submission', lambda text, n: pe_fisier(valideaza_submission, text, n, nume='s.csv'),
         [((H + '1,1,a\n1,2,b\n', 2), True), ((H + '1,1,a\n', 2), False)],
         ascunse=[(('id,datapointID,answer\n1,1,a\n', 1), False), ((H + '1,1,a\n1,3,b\n', 2), False),
                  ((',' + H + '0,1,1,a\n', 1), False), ((H + '1,1,\n', 1), False), ((H, 0), True),
                  ((H + '1,1,a\n1,2,b\n1,3,c\n', 2), False), (('', 0), False)],
         puncte=2, indiciu='citeste tot cu csv.reader, apoi verifica pe rand: antetul, len(randuri), fiecare rand')

### 8. Luminozitate cu clip

`lumineaza(img, delta)`: primeste o imagine `uint8` (gri sau color) si un intreg `delta` (pozitiv sau negativ). Intoarce imaginea cu `delta` adunat la fiecare pixel, **taiat la intervalul 0-255**, tot de tip `uint8`. Atentie la capcana din predare.

In [ ]:
def lumineaza(img, delta):
    pass  # TODO

In [ ]:
def t_lumineaza(img, delta):
    r = lumineaza(img, delta)
    return r.tolist(), str(r.dtype)

def ref_lumineaza(img, delta):
    r = np.clip(img.astype(int) + delta, 0, 255).astype(np.uint8)
    return r.tolist(), 'uint8'

a = np.array([[100, 200, 250]], dtype=np.uint8)
verifica('lumineaza', t_lumineaza,
         [((a, 10), ([[110, 210, 255]], 'uint8')), ((a, -150), ([[0, 50, 100]], 'uint8'))],
         ascunse=[((a, 0), ([[100, 200, 250]], 'uint8')), ((a, 1000), ([[255, 255, 255]], 'uint8'))],
         referinta=ref_lumineaza,
         generator=lambda: (np.random.randint(0, 256, (3, 4)).astype(np.uint8), np.random.randint(-300, 300)),
         puncte=2, indiciu='uint8 + numar se rostogoleste: converteste la int, np.clip(..., 0, 255), apoi .astype(np.uint8)')

### 9. Histograma de mana

`histograma(img)`: pentru o imagine gri `uint8`, intoarce o **lista** cu 256 de numere: cate pixeli au valoarea `0`, cati au `1`, ..., cati au `255`. Scrie tu bucla (un contor pe valoare), fara `np.bincount`.

In [ ]:
def histograma(img):
    pass  # TODO

In [ ]:
def ref_hist(img):
    return np.bincount(img.ravel(), minlength=256).tolist()

g = np.array([[0, 255, 0], [7, 7, 7]], dtype=np.uint8)
asteptat = [0] * 256
asteptat[0], asteptat[255], asteptat[7] = 2, 1, 3
verifica('histograma', histograma, [((g,), asteptat)],
         ascunse=[((np.zeros((2, 2), dtype=np.uint8),), [4] + [0] * 255)],
         referinta=ref_hist, generator=lambda: (np.random.randint(0, 256, (np.random.randint(1, 9), np.random.randint(1, 9))).astype(np.uint8),),
         puncte=2, indiciu='h = [0] * 256; for v in img.ravel(): h[v] += 1')

### 10. Alb-negru

`alb_negru(img, prag)`: pentru o imagine gri, intoarce o matrice `uint8` de aceeasi forma cu `255` unde pixelul e **strict** mai mare decat `prag` si `0` in rest. (Asta e imaginea din animatia `anim_prag`.)

In [ ]:
def alb_negru(img, prag):
    pass  # TODO

In [ ]:
def t_alb_negru(img, prag):
    r = alb_negru(img, prag)
    return r.tolist(), str(r.dtype)

g = np.array([[10, 200, 130], [90, 250, 40]], dtype=np.uint8)
verifica('alb_negru', t_alb_negru,
         [((g, 128), ([[0, 255, 255], [0, 255, 0]], 'uint8')), ((g, 255), ([[0, 0, 0], [0, 0, 0]], 'uint8'))],
         ascunse=[((g, 0), ([[255, 255, 255], [255, 255, 255]], 'uint8')), ((g, 40), ([[0, 255, 255], [255, 255, 0]], 'uint8'))],
         puncte=1, indiciu='(img > prag) e True/False; inmultit cu 255 devine 255/0; la final .astype(np.uint8)')

### 11. Provocare: blur 3 x 3

Un **blur** (estompare) inlocuieste fiecare pixel cu media vecinilor lui. Scrie `blur3(img)` pentru o matrice 2D (cel putin `3 x 3`): fiecare pixel **interior** devine media celor 9 valori din patratul `3 x 3` centrat pe el (pixelul insusi inclus); pixelii de pe margine raman neschimbati. Intoarce o matrice `float` noua, fara sa modifici `img`. Important: media se calculeaza pe valorile **originale**, nu pe cele deja estompate.

Poti face patru bucle (`i`, `j`, apoi vecinii) sau folosi slicing: suma celor 9 decupaje deplasate, impartita la 9.

In [ ]:
def blur3(img):
    pass  # TODO

In [ ]:
def ref_blur(img):
    a = np.asarray(img, dtype=float)
    r = a.copy()
    h, w = a.shape
    r[1:-1, 1:-1] = sum(a[i:h - 2 + i, j:w - 2 + j] for i in range(3) for j in range(3)) / 9
    return r

m = np.array([[0, 0, 0, 0], [0, 9, 0, 0], [0, 0, 0, 0]], dtype=float)
verifica('blur3', blur3, [((m,), np.array([[0, 0, 0, 0], [0, 1, 1, 0], [0, 0, 0, 0]], dtype=float))],
         ascunse=[((np.arange(9).reshape(3, 3),), np.array([[0, 1, 2], [3, 4, 5], [6, 7, 8]], dtype=float))],
         referinta=ref_blur,
         generator=lambda: (np.random.randint(0, 256, (np.random.randint(3, 8), np.random.randint(3, 8))),),
         puncte=3, indiciu='rezultatul se scrie in alt tablou (o copie), altfel vecinii urmatori vad valori deja estompate')
scor()